# Importing Libraries

In [1]:
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler, Subset
from torchvision import datasets, transforms
from collections import Counter
import numpy as np
from sklearn.model_selection import train_test_split
from torchvision.models import resnet18
from sklearn.metrics import f1_score, classification_report
from torchvision.models import mobilenet_v2

# Importing Dataset

In [2]:
root_dir = "dataset"
train_ratio = 0.8
batch_size = 32
num_workers = 4

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=root_dir)
targets = np.array(full_dataset.targets)
indices = np.arange(len(targets))

train_idx, val_idx = train_test_split(
    indices,
    test_size=1 - train_ratio,
    stratify=targets,
    random_state=42
)

train_dataset = Subset(datasets.ImageFolder(root=root_dir, transform=train_transform), train_idx)
val_dataset = Subset(datasets.ImageFolder(root=root_dir, transform=val_transform), val_idx)

train_targets = targets[train_idx]
class_counts = Counter(train_targets)
num_samples = len(train_dataset)

class_weights = {cls: num_samples / count for cls, count in class_counts.items()}
sample_weights = [class_weights[t] for t in train_targets]

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=num_samples, replacement=True)

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    sampler=sampler,
    num_workers=num_workers)
val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=num_workers)

print("Classes:", full_dataset.classes)
print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Train class counts:", class_counts)

Classes: ['battery', 'biological', 'cardboard', 'clothes', 'glass', 'leafs', 'metal', 'paper', 'plastic_bag', 'plastic_bottle']
Train size: 52669
Validation size: 13168
Train class counts: Counter({np.int64(1): 12689, np.int64(3): 8522, np.int64(8): 8000, np.int64(7): 7107, np.int64(4): 4450, np.int64(5): 3595, np.int64(2): 2626, np.int64(9): 2421, np.int64(6): 1748, np.int64(0): 1511})


# Model Definition

In [8]:
num_classes = len(full_dataset.classes)

model = mobilenet_v2(weights="IMAGENET1K_V1")
model.classifier[1] = torch.nn.Linear(model.last_channel, num_classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.features[18].parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss /= val_total
    val_acc = val_correct / val_total

    f1_macro = f1_score(all_labels, all_preds, average='macro')
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
        f"F1 (macro): {f1_macro:.4f}")

print("\nFinal classification report:")
print(classification_report(all_labels, all_preds, target_names=full_dataset.classes))


Epoch [1/10] Train Loss: 0.2867 | Train Acc: 0.9033 | Val Loss: 0.1525 | Val Acc: 0.9497 | F1 (macro): 0.9341
Epoch [2/10] Train Loss: 0.2804 | Train Acc: 0.9056 | Val Loss: 0.1400 | Val Acc: 0.9543 | F1 (macro): 0.9379
Epoch [3/10] Train Loss: 0.2777 | Train Acc: 0.9080 | Val Loss: 0.1261 | Val Acc: 0.9599 | F1 (macro): 0.9467
Epoch [4/10] Train Loss: 0.2715 | Train Acc: 0.9096 | Val Loss: 0.1368 | Val Acc: 0.9547 | F1 (macro): 0.9423
Epoch [5/10] Train Loss: 0.2700 | Train Acc: 0.9106 | Val Loss: 0.1379 | Val Acc: 0.9567 | F1 (macro): 0.9429
Epoch [6/10] Train Loss: 0.2663 | Train Acc: 0.9095 | Val Loss: 0.1272 | Val Acc: 0.9603 | F1 (macro): 0.9472
Epoch [7/10] Train Loss: 0.2723 | Train Acc: 0.9077 | Val Loss: 0.1314 | Val Acc: 0.9592 | F1 (macro): 0.9469
Epoch [8/10] Train Loss: 0.2622 | Train Acc: 0.9120 | Val Loss: 0.1446 | Val Acc: 0.9532 | F1 (macro): 0.9376
Epoch [9/10] Train Loss: 0.2582 | Train Acc: 0.9138 | Val Loss: 0.1247 | Val Acc: 0.9603 | F1 (macro): 0.9484
Epoch [10/

# Exporting the Model

In [17]:
torch.save(model.state_dict(), "model.pth")

In [18]:
dummy_input = torch.randn(1, 3, 224, 224).to(device)
model.eval()
torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=12
)

W1126 02:12:08.470000 29256 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 12).
Failed to convert the model to the target version 12 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "d:\Projects\radya\garbage-classification\venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Projects\radya\garbage-classification\venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "d:\Projects\radya\garbage-classification\venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  

Applied 104 of general pattern rewrite rules.


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.9.0+cu126',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,3,224,224]>
            ),
            outputs=(
                %"output"<FLOAT,[1,10]>
            ),
            initializers=(
                %"features.0.0.weight"<FLOAT,[32,3,3,3]>{Tensor(...)},
                %"features.1.conv.0.0.weight"<FLOAT,[32,1,3,3]>{Tensor(...)},
                %"features.1.conv.1.weight"<FLOAT,[16,32,1,1]>{Tensor(...)},
                %"features.2.conv.1.0.weight"<FLOAT,[96,1,3,3]>{Tensor(...)},
                %"classifier.1.bias"<FLOAT,[10]>{TorchTensor<FLOAT,[10]>(Parameter containing: tensor([ 0.0293,  0.0031, -0.0223,  0.0232,  0.0082,  0.0212,  0.0060, -0.0308, -0.0207,  0.0032], device='cuda:0', requires_g

# Validation

## PyTorch

In [9]:
num_classes = len(full_dataset.classes)

from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)

model.classifier[1] = torch.nn.Linear(
    model.classifier[1].in_features,
    num_classes
)

state = torch.load("model.pth", map_location="cpu")
model.load_state_dict(state)

model.eval()


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [10]:
import torch
from torchvision import transforms
from PIL import Image
import torch.nn.functional as F

def predict_image(model, image_path, class_names):
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0)

    model.eval()
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = F.softmax(outputs, dim=1)[0]

    results = {class_names[i]: float(probabilities[i]) for i in range(len(class_names))}

    return results


In [11]:
class_names = full_dataset.classes
result = predict_image(model, "test/battery1.jpg", class_names)

top_class = max(result, key=result.get)
print(f"Predicted: {top_class} ({result[top_class]*100:.2f}%)")

for cls, prob in result.items():
    print(f"{cls}: {prob*100:.2f}%")

Predicted: battery (99.56%)
battery: 99.56%
biological: 0.02%
cardboard: 0.01%
clothes: 0.03%
glass: 0.07%
leafs: 0.00%
metal: 0.00%
paper: 0.28%
plastic_bag: 0.02%
plastic_bottle: 0.00%


In [12]:
import os

correct = 0
total = 0

for filename in os.listdir("test"):
    if not filename.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    image_path = os.path.join("test", filename)

    true_label = ''.join([c for c in filename if not c.isdigit()]).split('.')[0]

    result = predict_image(model, image_path, class_names)
    predicted_label = max(result, key=result.get)

    is_correct = predicted_label.lower() == true_label.lower()
    total += 1
    correct += int(is_correct)

    print(f"{filename}: predicted {predicted_label} ({result[predicted_label]*100:.2f}%) | true: {true_label} | {'✓' if is_correct else '✗'}")

accuracy = (correct / total) * 100 if total > 0 else 0
print(f"\nAccuracy: {accuracy:.2f}%")


battery1.jpg: predicted battery (99.56%) | true: battery | ✓
biological1.jpg: predicted biological (99.99%) | true: biological | ✓
cardboard1.jpg: predicted cardboard (98.97%) | true: cardboard | ✓
clothes1.jpg: predicted clothes (99.85%) | true: clothes | ✓
glass1.jpg: predicted glass (80.96%) | true: glass | ✓
leafs1.jpg: predicted biological (99.83%) | true: leafs | ✗
leafs2.jpg: predicted biological (99.38%) | true: leafs | ✗
leafs3.jpg: predicted biological (99.80%) | true: leafs | ✗
metal1.jpg: predicted metal (99.89%) | true: metal | ✓
paper1.jpg: predicted paper (69.99%) | true: paper | ✓
plastic_bag1.jpg: predicted plastic_bottle (99.98%) | true: plastic_bag | ✗
plastic_bag2.jpg: predicted plastic_bag (99.92%) | true: plastic_bag | ✓
plastic_bottle1.jpg: predicted plastic_bottle (100.00%) | true: plastic_bottle | ✓

Accuracy: 69.23%


## ONNX

In [3]:
import onnxruntime as ort

session = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])

In [4]:
from PIL import Image
import numpy as np
import torchvision.transforms as T

transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

def preprocess(img):
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    w, h = img.size
    if w < h:
        new_w = 256
        new_h = int(h * 256 / w)
    else:
        new_h = 256
        new_w = int(w * 256 / h)
    img = img.resize((new_w, new_h), Image.BILINEAR)
    left = (new_w - 224) // 2
    top = (new_h - 224) // 2
    right = left + 224
    bottom = top + 224
    img = img.crop((left, top, right, bottom))
    arr = np.asarray(img).astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    arr = (arr - mean) / std
    arr = arr.transpose(2, 0, 1)
    arr = np.expand_dims(arr, axis=0)

    return arr

def predict_image_onnx(session, image_path, class_names):
    img = Image.open(image_path).convert("RGB")
    tensor = preprocess(img)

    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    outputs = session.run([output_name], {input_name: tensor})[0][0]

    exp = np.exp(outputs - np.max(outputs))
    probs = exp / exp.sum()

    result = {class_names[i]: float(probs[i]) for i in range(len(class_names))}
    return result

In [18]:
correct = 0
total = 0

for filename in os.listdir("test"):
    if not filename.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    image_path = os.path.join("test", filename)

    true_label = ''.join([c for c in filename if not c.isdigit()]).split('.')[0]
    class_names = full_dataset.classes
    result = predict_image_onnx(session, image_path, class_names)
    predicted_label = max(result, key=result.get)

    is_correct = predicted_label.lower() == true_label.lower()
    total += 1
    correct += int(is_correct)

    print(f"{filename}: predicted {predicted_label} ({result[predicted_label]*100:.2f}%) | true: {true_label} | {'✓' if is_correct else '✗'}")

accuracy = (correct / total) * 100 if total > 0 else 0
print(f"\nAccuracy: {accuracy:.2f}%")

battery1.jpg: predicted battery (99.56%) | true: battery | ✓
biological1.jpg: predicted biological (99.99%) | true: biological | ✓
cardboard1.jpg: predicted cardboard (98.97%) | true: cardboard | ✓
clothes1.jpg: predicted clothes (99.71%) | true: clothes | ✓
glass1.jpg: predicted glass (80.96%) | true: glass | ✓
leafs1.jpg: predicted biological (99.83%) | true: leafs | ✗
leafs2.jpg: predicted biological (99.38%) | true: leafs | ✗
leafs3.jpg: predicted biological (99.80%) | true: leafs | ✗
metal1.jpg: predicted metal (99.89%) | true: metal | ✓
paper1.jpg: predicted paper (69.99%) | true: paper | ✓
plastic_bag1.jpg: predicted plastic_bottle (99.98%) | true: plastic_bag | ✗
plastic_bag2.jpg: predicted plastic_bag (99.92%) | true: plastic_bag | ✓
plastic_bottle1.jpg: predicted plastic_bottle (100.00%) | true: plastic_bottle | ✓

Accuracy: 69.23%


In [14]:
def predict_single_image(session, image_path, class_names):
    result = predict_image_onnx(session, image_path, class_names)

    predicted_label = max(result, key=result.get)
    confidence = result[predicted_label]

    print(f"Image: {os.path.basename(image_path)}")
    print(f"Predicted: {predicted_label} ({confidence*100:.2f}%)\n")
    print("All class confidences:")
    for cls, prob in result.items():
        print(f"  {cls}: {prob*100:.2f}%")

    return predicted_label, confidence, result

In [20]:
pred_label, conf, conf_dict = predict_single_image(
    session,
    "test/cardboard1.jpg",
    full_dataset.classes
)

Image: cardboard1.jpg
Predicted: cardboard (98.97%)

All class confidences:
  battery: 0.01%
  biological: 1.00%
  cardboard: 98.97%
  clothes: 0.00%
  glass: 0.00%
  leafs: 0.00%
  metal: 0.00%
  paper: 0.01%
  plastic_bag: 0.00%
  plastic_bottle: 0.00%
